In [1]:
# ============================================
# GOLD LAYER - Star Schema
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder \
    .appName("Ecommerce-Gold-Layer") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Started Successfully")

Spark Session Started Successfully


In [10]:
# 1. Register the DataFrames as SQL Temp Views
import pandas as pd

# 1. Register the DataFrames as SQL Temp Views

fact_order_items = spark.read.parquet(
    "gold/fact_order_items"
)

dim_customers = spark.read.parquet(
    "gold/dim_customers"
)

# Pandas DataFrame for customer analytics
fact_df = pd.read_parquet(
    "gold/fact_order_items"
)

fact_order_items.createOrReplaceTempView(
    "fact_order_items"
)

dim_customers.createOrReplaceTempView(
    "dim_customers"
)

# 2. Run the EDA Query
frequency_eda_sql = """
WITH customer_freq AS (
    SELECT 
        c.customer_unique_id, 
        COUNT(DISTINCT f.order_id) AS frequency
    FROM fact_order_items f
    JOIN dim_customers c ON f.customer_id = c.customer_id
    WHERE f.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT    
    frequency,    
    COUNT(*) AS num_customers,    
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_of_customers
FROM customer_freq
GROUP BY frequency
ORDER BY frequency;
"""

print("Frequency Distribution:")
spark.sql(frequency_eda_sql).show(20)

Frequency Distribution:
+---------+-------------+----------------+
|frequency|num_customers|pct_of_customers|
+---------+-------------+----------------+
|        1|        90557|           97.00|
|        2|         2573|            2.76|
|        3|          181|            0.19|
|        4|           28|            0.03|
|        5|            9|            0.01|
|        6|            5|            0.01|
|        7|            3|            0.00|
|        9|            1|            0.00|
|       15|            1|            0.00|
+---------+-------------+----------------+



In [9]:
# ============================================
# MART 1: SELLER PERFORMANCE & RISK
# Seller-order grain → Seller-level scorecard
# ============================================

import os

os.makedirs("analytics_exports", exist_ok=True)

seller_risk_sql = """
WITH seller_orders AS (

    -- First reduce item-level data to seller + order grain
    SELECT
        seller_id,
        order_id,

        -- Revenue remains item-level and is summed into order revenue
        ROUND(SUM(price), 2) AS order_revenue,

        -- These attributes should be consistent across items
        MAX(order_status) AS order_status,
        MAX(delivery_days) AS delivery_days,

        -- One flag per seller-order
        MAX(
            CASE
                WHEN is_late = TRUE THEN 1
                ELSE 0
            END
        ) AS is_late

    FROM fact_order_items

    GROUP BY
        seller_id,
        order_id
),

seller_metrics AS (

    -- Now calculate seller-level performance metrics
    SELECT
        seller_id,

        COUNT(*) AS total_orders,

        ROUND(SUM(order_revenue), 2) AS total_revenue,

        ROUND(AVG(delivery_days), 1) AS avg_delivery_days,

        -- Late delivery rate
        ROUND(
            100.0 * SUM(is_late) / COUNT(*),
            2
        ) AS late_rate_pct,

        -- Cancellation rate
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN order_status = 'canceled' THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS cancel_rate_pct

    FROM seller_orders

    GROUP BY seller_id
),

seller_scores AS (

    SELECT
        seller_id,
        total_orders,
        total_revenue,
        avg_delivery_days,
        late_rate_pct,
        cancel_rate_pct,

        -- Composite Risk Index
        -- Cancellation is weighted 2x late delivery
        ROUND(
            late_rate_pct
            + (cancel_rate_pct * 2.0),
            2
        ) AS composite_risk_score

    FROM seller_metrics
)

SELECT
    seller_id,
    total_orders,
    total_revenue,
    avg_delivery_days,
    late_rate_pct,
    cancel_rate_pct,
    composite_risk_score,

    CASE
        WHEN composite_risk_score >= 60 THEN 'High'
        WHEN composite_risk_score >= 40 THEN 'Medium'
        ELSE 'Low'
    END AS risk_category

FROM seller_scores

-- Avoid unstable results from very low-volume sellers
WHERE total_orders >= 10

ORDER BY composite_risk_score DESC
"""

# Run the query
seller_scorecard = spark.sql(seller_risk_sql)

# Preview
print("Seller Risk Scorecard Preview:")
seller_scorecard.show(20, truncate=False)

# Export a single CSV file for dashboard use
seller_scorecard.toPandas().to_csv(
    "analytics_exports/seller_scorecard.csv",
    index=False
)

print("Seller CSV created for dashboard.")




Seller Risk Scorecard Preview:
+--------------------------------+------------+-------------+-----------------+-------------+---------------+--------------------+-------------+
|seller_id                       |total_orders|total_revenue|avg_delivery_days|late_rate_pct|cancel_rate_pct|composite_risk_score|risk_category|
+--------------------------------+------------+-------------+-----------------+-------------+---------------+--------------------+-------------+
|81783131d2a97c8d44d406a4be81b5d9|13          |1782.54      |7.0              |7.69         |38.46          |84.61               |High         |
|b1b3948701c5c72445495bd161b83a4c|18          |24699.19     |23.4             |50.00        |11.11          |72.22               |High         |
|973f21788dfab357250f69a8dcb7ddee|10          |909.0        |23.6             |50.00        |10.00          |70.00               |High         |
|20b54c376b794ed028df09a3cd88e8dc|11          |1342.95      |21.1             |18.18        |18.18 

In [11]:
import pandas as pd
import os

print("\nGenerating Customer Segmentation via Pandas...")

# Read Gold customer data
cust_df = pd.read_parquet("gold/dim_customers")

# ------------------------------------------------
# STEP 1: Keep only delivered orders
# ------------------------------------------------

df = (
    fact_df[
        fact_df['order_status'] == 'delivered'
    ]
    .merge(
        cust_df,
        on='customer_id'
    )
)

# Ensure datetime format
df['order_purchase_timestamp'] = pd.to_datetime(
    df['order_purchase_timestamp']
)

# ------------------------------------------------
# STEP 2: Customer-level metrics
# ------------------------------------------------

cust_base = (
    df.groupby('customer_unique_id')
    .agg(
        frequency=('order_id', 'nunique'),
        monetary=('price', 'sum'),
        last_purchase_date=('order_purchase_timestamp', 'max')
    )
    .reset_index()
)

# ------------------------------------------------
# STEP 3: Recency
# ------------------------------------------------

max_date = df['order_purchase_timestamp'].max()

cust_base['recency_days'] = (
    max_date - cust_base['last_purchase_date']
).dt.days

# ------------------------------------------------
# STEP 4: Monetary score
# Quartiles: 1 = lowest, 4 = highest
# ------------------------------------------------

cust_base['m_score'] = (
    pd.qcut(
        cust_base['monetary'].rank(method='first'),
        4,
        labels=[1, 2, 3, 4]
    )
    .astype(int)
)

# ------------------------------------------------
# STEP 5: Recency score
# 4 = most recent
# 1 = least recent
# ------------------------------------------------

cust_base['r_score'] = (
    pd.qcut(
        cust_base['recency_days'].rank(method='first'),
        4,
        labels=[4, 3, 2, 1]
    )
    .astype(int)
)

# ------------------------------------------------
# STEP 6: Business segmentation
# ------------------------------------------------

def assign_segment(row):

    # Customers who purchased more than once
    if row['frequency'] > 1:
        return 'Repeat Loyalists'

    # One-time + high value + recent
    elif (
        row['frequency'] == 1
        and row['m_score'] >= 3
        and row['r_score'] >= 3
    ):
        return 'Recent High-Value'

    # One-time + high value + not recent
    elif (
        row['frequency'] == 1
        and row['m_score'] >= 3
        and row['r_score'] <= 2
    ):
        return 'Lapsed High-Value'

    # Everything else
    else:
        return 'Low-Value One-Time'


cust_base['segment_name'] = cust_base.apply(
    assign_segment,
    axis=1
)

# Round monetary value
cust_base['monetary'] = round(
    cust_base['monetary'],
    2
)

# ------------------------------------------------
# STEP 7: Final output
# ------------------------------------------------

final_segments = cust_base[
    [
        'customer_unique_id',
        'recency_days',
        'frequency',
        'monetary',
        'r_score',
        'm_score',
        'segment_name'
    ]
].sort_values(
    'monetary',
    ascending=False
)

# ------------------------------------------------
# STEP 8: Export
# ------------------------------------------------

os.makedirs(
    "analytics_exports",
    exist_ok=True
)

final_segments.to_csv(
    "analytics_exports/customer_segmentation.csv",
    index=False
)

print("Mart 2: Customer Segmentation Exported!")

print("\nSegment Distribution:")
print(
    final_segments['segment_name']
    .value_counts()
)

print("\nRevenue by Segment:")
print(
    final_segments
    .groupby('segment_name')['monetary']
    .sum()
    .sort_values(ascending=False)
)


Generating Customer Segmentation via Pandas...
Mart 2: Customer Segmentation Exported!

Segment Distribution:
segment_name
Low-Value One-Time    46221
Recent High-Value     22292
Lapsed High-Value     22044
Repeat Loyalists       2801
Name: count, dtype: int64

Revenue by Segment:
segment_name
Recent High-Value     5163841.94
Lapsed High-Value     5151392.18
Low-Value One-Time    2177855.24
Repeat Loyalists       728408.75
Name: monetary, dtype: float64


In [12]:
print("\nGenerating Customer Cohort Retention via Pandas...")

# ------------------------------------------------
# STEP 1: Read Gold data
# ------------------------------------------------

cust_df = pd.read_parquet("gold/dim_customers")

# Merge fact table with customer dimension
df = (
    fact_df[
        fact_df['order_status'] == 'delivered'
    ]
    .merge(
        cust_df,
        on='customer_id'
    )
)

# Ensure datetime
df['order_purchase_timestamp'] = pd.to_datetime(
    df['order_purchase_timestamp']
)

# ------------------------------------------------
# STEP 2: Create order-level customer dataset
# ------------------------------------------------
# One customer can have multiple item rows for
# the same order, so first reduce to one row/order.

customer_orders = (
    df.groupby(
        [
            'customer_unique_id',
            'order_id',
            'order_purchase_timestamp'
        ],
        as_index=False
    )
    .agg(
        order_revenue=('price', 'sum')
    )
)

# Convert purchase date to month
customer_orders['purchase_month'] = (
    customer_orders['order_purchase_timestamp']
    .dt.to_period('M')
)

# ------------------------------------------------
# STEP 3: Determine each customer's first month
# ------------------------------------------------

customer_first = (
    customer_orders
    .groupby('customer_unique_id')['purchase_month']
    .min()
    .reset_index()
    .rename(
        columns={
            'purchase_month': 'cohort_month'
        }
    )
)

# Attach cohort month to every customer purchase
customer_orders = customer_orders.merge(
    customer_first,
    on='customer_unique_id',
    how='left'
)

# ------------------------------------------------
# STEP 4: Calculate months since first purchase
# ------------------------------------------------

customer_orders['cohort_index'] = (
    (customer_orders['purchase_month'].dt.year
     - customer_orders['cohort_month'].dt.year) * 12
    +
    (customer_orders['purchase_month'].dt.month
     - customer_orders['cohort_month'].dt.month)
)

# ------------------------------------------------
# STEP 5: Count active customers
# ------------------------------------------------
# Distinct customers in each cohort/month

cohort_data = (
    customer_orders
    .groupby(
        ['cohort_month', 'cohort_index']
    )['customer_unique_id']
    .nunique()
    .reset_index()
    .rename(
        columns={
            'customer_unique_id': 'active_customers'
        }
    )
)

# ------------------------------------------------
# STEP 6: Determine original cohort size
# ------------------------------------------------

cohort_sizes = (
    cohort_data[
        cohort_data['cohort_index'] == 0
    ][
        ['cohort_month', 'active_customers']
    ]
    .rename(
        columns={
            'active_customers': 'cohort_size'
        }
    )
)

# Attach cohort size
cohort_data = cohort_data.merge(
    cohort_sizes,
    on='cohort_month',
    how='left'
)

# ------------------------------------------------
# STEP 7: Calculate retention %
# ------------------------------------------------

cohort_data['retention_pct'] = round(
    cohort_data['active_customers']
    / cohort_data['cohort_size']
    * 100,
    2
)

# ------------------------------------------------
# STEP 8: Format cohort month
# ------------------------------------------------

cohort_data['cohort_month'] = (
    cohort_data['cohort_month']
    .astype(str)
)

# ------------------------------------------------
# STEP 9: Export long-format cohort mart
# ------------------------------------------------

os.makedirs(
    "analytics_exports",
    exist_ok=True
)

cohort_data = cohort_data[
    [
        'cohort_month',
        'cohort_index',
        'cohort_size',
        'active_customers',
        'retention_pct'
    ]
].sort_values(
    ['cohort_month', 'cohort_index']
)

cohort_data.to_csv(
    "analytics_exports/cohort_retention.csv",
    index=False
)

print("Mart 3: Cohort Retention Exported!")

print("\nCohort Retention Preview:")
print(cohort_data.head(20))


Generating Customer Cohort Retention via Pandas...
Mart 3: Cohort Retention Exported!

Cohort Retention Preview:
   cohort_month  cohort_index  cohort_size  active_customers  retention_pct
0       2016-09             0            1                 1         100.00
1       2016-10             0          262               262         100.00
2       2016-10             6          262                 1           0.38
3       2016-10             9          262                 1           0.38
4       2016-10            11          262                 1           0.38
5       2016-10            13          262                 1           0.38
6       2016-10            15          262                 1           0.38
7       2016-10            17          262                 1           0.38
8       2016-10            19          262                 2           0.76
9       2016-10            20          262                 2           0.76
10      2016-12             0            1        

In [13]:
print("\nMonth-level retention summary:")

print(
    cohort_data[
        cohort_data['cohort_index'] > 0
    ]
    .groupby('cohort_index')
    .agg(
        cohorts=('cohort_month', 'nunique'),
        total_active_customers=('active_customers', 'sum'),
        avg_retention_pct=('retention_pct', 'mean')
    )
    .reset_index()
    .head(12)
)


Month-level retention summary:
    cohort_index  cohorts  total_active_customers  avg_retention_pct
0              1       20                     420           5.450500
1              2       18                     274           0.337778
2              3       17                     191           0.248235
3              4       16                     176           0.288750
4              5       15                     141           0.232667
5              6       15                     129           0.269333
6              7       13                     102           0.211538
7              8       12                      86           0.207500
8              9       11                      60           0.191818
9             10       10                      75           0.265000
10            11       10                      57           0.231000
11            12        8                      36           0.212500


In [14]:
# ============================================
# Dashboard Summary Exports
# ============================================

import pandas as pd
import os

os.makedirs("analytics_exports/dashboard", exist_ok=True)

# --------------------------------------------
# 1. Customer Segment Summary
# --------------------------------------------

segment_summary = (
    final_segments
    .groupby('segment_name')
    .agg(
        customers=('customer_unique_id', 'nunique'),
        revenue=('monetary', 'sum')
    )
    .reset_index()
)

segment_summary['customer_share_pct'] = round(
    segment_summary['customers']
    / segment_summary['customers'].sum()
    * 100,
    2
)

segment_summary['revenue_share_pct'] = round(
    segment_summary['revenue']
    / segment_summary['revenue'].sum()
    * 100,
    2
)

segment_summary.to_csv(
    "analytics_exports/dashboard/segment_summary.csv",
    index=False
)


# --------------------------------------------
# 2. Seller Risk Summary
# --------------------------------------------

# Read the already-exported seller mart
seller_data = pd.read_csv(
    "analytics_exports/seller_scorecard.csv"
)

seller_summary = (
    seller_data
    .sort_values(
        'composite_risk_score',
        ascending=False
    )
    .head(10)
)

seller_summary.to_csv(
    "analytics_exports/dashboard/top_seller_risk.csv",
    index=False
)


# --------------------------------------------
# 3. Cohort Retention Summary
# --------------------------------------------

cohort_summary = (
    cohort_data[
        cohort_data['cohort_index'] > 0
    ]
    .groupby('cohort_index')
    .agg(
        total_active_customers=('active_customers', 'sum'),
        total_cohort_customers=('cohort_size', 'sum'),
        cohorts=('cohort_month', 'nunique')
    )
    .reset_index()
)

cohort_summary['retention_pct'] = round(
    cohort_summary['total_active_customers']
    / cohort_summary['total_cohort_customers']
    * 100,
    2
)

cohort_summary.to_csv(
    "analytics_exports/dashboard/cohort_summary.csv",
    index=False
)


print("Dashboard summary files created successfully.")

print("\nSegment Summary:")
print(segment_summary)

print("\nTop Seller Risk:")
print(seller_summary)

print("\nCohort Summary:")
print(cohort_summary.head(12))

Dashboard summary files created successfully.

Segment Summary:
         segment_name  customers     revenue  customer_share_pct  \
0   Lapsed High-Value      22044  5151392.18               23.61   
1  Low-Value One-Time      46221  2177855.24               49.51   
2   Recent High-Value      22292  5163841.94               23.88   
3    Repeat Loyalists       2801   728408.75                3.00   

   revenue_share_pct  
0              38.96  
1              16.47  
2              39.06  
3               5.51  

Top Seller Risk:
                          seller_id  total_orders  total_revenue  \
0  81783131d2a97c8d44d406a4be81b5d9            13        1782.54   
1  b1b3948701c5c72445495bd161b83a4c            18       24699.19   
2  973f21788dfab357250f69a8dcb7ddee            10         909.00   
3  20b54c376b794ed028df09a3cd88e8dc            11        1342.95   
4  7bcd7c5f8631701474db233ccf1c094b            11         666.00   
5  4c8b8048e33af2bf94f2eb547746a916            23     